# Iceberg Metadata & Statistics via Trino

This notebook queries Iceberg metadata tables through **Trino**.  
Trino exposes Iceberg hidden metadata tables using the `$` suffix — the table name and suffix must be quoted together as a single identifier:

```sql
-- correct
SELECT * FROM iceberg.demo."transactions$history"

-- wrong — $ is not a valid identifier character outside quotes
SELECT * FROM iceberg.demo.transactions.$history

```

For more details about availble metadata tables in trino **check** the **[link](https://trino.io/docs/current/connector/iceberg.html#metadata-tables)**

| Metadata Table | What it shows |
|---|---|
| `"table$history"` | Every snapshot commit — lineage |
| `"table$snapshots"` | Snapshot details: operation, file/record counts |
| `"table$files"` | Current data files with column-level stats |
| `"table$manifests"` | Manifest files that group data files |
| `"table$partitions"` | Partition-level record/file counts |
| `"table$refs"` | Named references (branches / tags) |

**Prerequisites:** Docker Compose stack must be running (`docker compose up -d`)  
Trino coordinator is available at `http://trino-coordinator:8080`

---
## 0. Setup: connect to Trino
---

In [1]:
import trino
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

conn = trino.dbapi.connect(
    host="trino-coordinator",
    port=8080,
    user="jupyter",
    catalog="iceberg",
    schema="demo",
)

def query(sql: str) -> pd.DataFrame:
    """Run a Trino SQL query and return a DataFrame."""
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description] if cur.description else []
    return pd.DataFrame(rows, columns=cols)

def meta(table: str, suffix: str) -> str:
    """Return the fully-qualified Trino metadata table reference.
    
    Trino requires the table name and $suffix to be quoted together as a
    single identifier, e.g.  iceberg.demo."transactions$history"
    """
    # table is just the bare name, e.g. 'transactions'
    return f'iceberg.demo."{table}${suffix}"'

# Verify connection and list available tables
print("Connected to Trino. Tables in iceberg.demo:")
query("SHOW TABLES IN iceberg.demo")

Connected to Trino. Tables in iceberg.demo:


,Table
0,transactions
1,users


---
## 1. Select a table
---

Choose which Iceberg table to inspect. All metadata sections below will use the selected table.

In [2]:
available_tables = ["transactions", "users"]

table_selector = widgets.Dropdown(
    options=available_tables,
    value=available_tables[0],
    description="Table:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

display(table_selector)

Dropdown(description='Table:', layout=Layout(width='400px'), options=('transactions', 'users'), style=Descript…

In [4]:
# Re-run this cell after changing the dropdown above
TABLE = table_selector.value
FULL_TABLE = f"iceberg.demo.{TABLE}"
print(f"Inspecting: {FULL_TABLE}")

Inspecting: iceberg.demo.transactions


---
## 2. Select table Properties
---
Table properies store basic Iceberg-related configration settings on the table level

In [5]:
# show the table properties
query(f"""
SELECT * FROM {meta(TABLE, 'properties')}
"""
)

,key,value
0,created-at,2026-05-22T18:26:11.427220625Z
1,write.format.default,parquet
2,write.metadata.previous-versions-max,2147483647
3,write.parquet.compression-codec,zstd
4,write.upsert.enabled,false


---
## 3. History (`$history`)
---


The `$history` metadata table tracks every snapshot committed to the table.

| Column | Type | Description |
|---|---|---|
| `made_current_at` | timestamp | When this snapshot became current |
| `snapshot_id` | bigint | Unique snapshot identifier |
| `parent_id` | bigint | Parent snapshot ID (null for first) |
| `is_current_ancestor` | boolean | On the current lineage chain? |

The [$history](https://trino.io/docs/current/connector/iceberg.html#history-table) table provides a log of the metadata changes performed on the Iceberg table. 

You can retrieve the changelog of the Iceberg table test_table by using the following query:

In [6]:
history_df = query(f"""
    SELECT
         *
    FROM {meta(TABLE, 'history')}

""")

print(f"Total snapshots in history: {len(history_df)}")
history_df

Total snapshots in history: 5


,made_current_at,snapshot_id,parent_id,is_current_ancestor
0,2026-05-22 18:26:32.666000+00:00,5200535123694019209,NaN,True
1,2026-05-22 18:27:03.629000+00:00,5170620940160670048,5.200535e+18,True
2,2026-05-22 18:27:32.682000+00:00,2524636529417930839,5.170621e+18,True
3,2026-05-22 18:29:32.650000+00:00,1381667802231900743,2.524637e+18,True
4,2026-05-22 18:30:02.498000+00:00,1086209458475638151,1.381668e+18,True


## 3.1 time travel with snapshot id
Given the snapshot id we can query the table as of specific time/snapshot, practically speaking making a time travel.

In [7]:
# Time-travel: read the table AS OF a specific snapshot
if not history_df.empty:
    oldest_snapshot_id = history_df["snapshot_id"].iloc[-1]
    print(f"Reading {FULL_TABLE} AS OF snapshot {oldest_snapshot_id} (oldest):")
    display(query(f"""
        SELECT *
        FROM {FULL_TABLE}
        FOR VERSION AS OF {oldest_snapshot_id}
        LIMIT 5
    """))
else:
    print("No history found.")

Reading iceberg.demo.transactions AS OF snapshot 1086209458475638151 (oldest):


,transaction_id,user_id,amount,currency,type,status,event_time
0,txn-7445889,user-027,289.13,CAD,TRANSFER,FAILED,2026-05-22 11:44:43.768
1,txn-7445890,user-024,1832.36,GBP,WITHDRAWAL,PENDING,2026-05-22 11:44:43.771
2,txn-7445891,user-040,1960.61,GBP,DEPOSIT,COMPLETED,2026-05-22 11:44:43.774
3,txn-7445892,user-026,470.23,USD,WITHDRAWAL,COMPLETED,2026-05-22 11:44:43.777
4,txn-7445893,user-007,788.56,EUR,TRANSFER,PENDING,2026-05-22 11:44:43.780


---
## 4. Snapshots (`$snapshots`)

The `$snapshots` table reveals the **content** of each snapshot — operation type, manifest counts, and record/file deltas via the `summary` map.

| Column | Type | Description |
|---|---|---|
| `committed_at` | timestamp | Wall-clock time of the commit |
| `snapshot_id` | bigint | Unique snapshot ID |
| `parent_id` | bigint | Parent snapshot ID |
| `operation` | string | `append`, `overwrite`, `replace`, `delete` |
| `manifest_list` | string | Path to the manifest-list Avro file |
| `summary` | map | Key metrics: added/deleted records, data files |

In [8]:
# All snapshots — most recent first
snapshots_df = query(f"""
    SELECT
        *
    FROM {meta(TABLE, 'snapshots')}
    ORDER BY committed_at DESC
""")

print(f"Snapshots: {len(snapshots_df)}")
snapshots_df

Snapshots: 5


,committed_at,snapshot_id,parent_id,operation,manifest_list,summary
0,2026-05-22 18:30:02.498000+00:00,1086209458475638151,1.381668e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
1,2026-05-22 18:29:32.650000+00:00,1381667802231900743,2.524637e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
2,2026-05-22 18:27:32.682000+00:00,2524636529417930839,5.170621e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
3,2026-05-22 18:27:03.629000+00:00,5170620940160670048,5.200535e+18,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...
4,2026-05-22 18:26:32.666000+00:00,5200535123694019209,NaN,append,s3://iceberg-warehouse/demo/transactions/metad...,{'flink.operator-id': 'cf155f65686cb012844f7c7...


In [9]:
# Extract key metrics from the summary map
# summary is a map<varchar,varchar> — use element_at() to pull out individual keys
query(f"""
    SELECT
        committed_at,
        snapshot_id,
        operation,
        CAST(element_at(summary, 'added-records')         AS bigint) AS added_records,
        CAST(element_at(summary, 'deleted-records')       AS bigint) AS deleted_records,
        CAST(element_at(summary, 'added-data-files')      AS bigint) AS added_data_files,
        CAST(element_at(summary, 'removed-data-files')    AS bigint) AS removed_data_files,
        CAST(element_at(summary, 'total-records')         AS bigint) AS total_records,
        CAST(element_at(summary, 'total-data-files')      AS bigint) AS total_data_files
    FROM {meta(TABLE, 'snapshots')}
    ORDER BY committed_at DESC
""")

,committed_at,snapshot_id,operation,added_records,deleted_records,added_data_files,removed_data_files,total_records,total_data_files
0,2026-05-22 18:30:02.498000+00:00,1086209458475638151,append,14800,None,1,None,10728185,5
1,2026-05-22 18:29:32.650000+00:00,1381667802231900743,append,3600,None,1,None,10713385,4
2,2026-05-22 18:27:32.682000+00:00,2524636529417930839,append,3147225,None,1,None,10709785,3
3,2026-05-22 18:27:03.629000+00:00,5170620940160670048,append,6403928,None,1,None,7562560,2
4,2026-05-22 18:26:32.666000+00:00,5200535123694019209,append,1158632,None,1,None,1158632,1


---
## 5. Files (`$files`)
---

The `$files` metadata table lists every **current data file** with column-level statistics. This is the key table for diagnosing data skew, small-file problems, and partition layout.

| Column | Type | Description |
|---|---|---|
| `content` | int | 0 = DATA, 1 = POSITION_DELETES, 2 = EQUALITY_DELETES |
| `file_path` | string | Full S3 path |
| `file_format` | string | `PARQUET`, `ORC`, `AVRO` |
| `record_count` | bigint | Rows in this file |
| `file_size_in_bytes` | bigint | Compressed file size |
| `column_sizes` | map | Bytes used per column field ID |
| `value_counts` | map | Non-null values per field ID |
| `null_value_counts` | map | Null values per field ID |
| `lower_bounds` | map | Min value per field ID |
| `upper_bounds` | map | Max value per field ID |

## 5.1 Let's show a sample of file level stats

In [10]:
# All current data files — most rows first
files_df = query(f"""
    SELECT
         *
    FROM {meta(TABLE, 'files')}
    ORDER BY record_count DESC
""")

print(f"Total data files: {len(files_df)}")
files_df[:5]
# files_df[["content", "file_format", "record_count", "file_size_in_bytes", "file_path"]]

Total data files: 43


,content,file_path,file_format,spec_id,record_count,file_size_in_bytes,column_sizes,value_counts,null_value_counts,nan_value_counts,lower_bounds,upper_bounds,key_metadata,split_offsets,equality_ids,sort_order_id,readable_metrics
0,0,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,0,6403928,47506823,"{1: 2608663, 2: 4831943, 3: 21480801, 4: 21866...","{1: 6403928, 2: 6403928, 3: 6403928, 4: 640392...","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}",{3: 0},"{1: 'txn-1040996', 2: 'user-001', 3: '1.0', 4:...","{1: 'txn-7445888', 2: 'user-050', 3: '2000.0',...",None,[4],None,0,"{""amount"":{""column_size"":21480801,""value_count..."
1,0,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,0,3147225,23517795,"{1: 1458224, 2: 2374771, 3: 10557360, 4: 10746...","{1: 3147225, 2: 3147225, 3: 3147225, 4: 314722...","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}",{3: 0},"{1: 'txn-10000000', 2: 'user-001', 3: '1.0', 4...","{1: 'txn-9999999', 2: 'user-050', 3: '2000.0',...",None,[4],None,0,"{""amount"":{""column_size"":10557360,""value_count..."
2,0,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,0,1158632,8723784,"{1: 486079, 2: 874350, 3: 3885578, 4: 395572, ...","{1: 1158632, 2: 1158632, 3: 1158632, 4: 115863...","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}",{3: 0},"{1: 'txn-000001', 2: 'user-001', 3: '1.0', 4: ...","{1: 'txn-999999', 2: 'user-050', 3: '2000.0', ...",None,[4],None,0,"{""amount"":{""column_size"":3885578,""value_count""..."
3,0,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,0,14850,92335,"{1: 7245, 2: 11388, 3: 49771, 4: 5183, 5: 5198...","{1: 14850, 2: 14850, 3: 14850, 4: 14850, 5: 14...","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}",{3: 0},"{1: 'txn-373501', 2: 'user-001', 3: '1.17', 4:...","{1: 'txn-388350', 2: 'user-050', 3: '1999.95',...",None,[4],None,0,"{""amount"":{""column_size"":49771,""value_count"":1..."
4,0,s3://iceberg-warehouse/demo/transactions/data/...,PARQUET,0,14850,90993,"{1: 7333, 2: 11386, 3: 49809, 4: 5162, 5: 5194...","{1: 14850, 2: 14850, 3: 14850, 4: 14850, 5: 14...","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}",{3: 0},"{1: 'txn-181201', 2: 'user-001', 3: '1.22', 4:...","{1: 'txn-196050', 2: 'user-050', 3: '1999.5', ...",None,[4],None,0,"{""amount"":{""column_size"":49809,""value_count"":1..."


## 5.2 Let's check if we have a small files issue

In [11]:
# File-level size statistics — spot small-file problems
query(f"""
    SELECT
        file_format,
        count(*)                                            AS file_count,
        sum(record_count)                                   AS total_records,
        round(avg(record_count), 0)                         AS avg_records_per_file,
        min(record_count)                                   AS min_records,
        max(record_count)                                   AS max_records,
        sum(file_size_in_bytes)                             AS total_size_bytes,
        round(avg(file_size_in_bytes) / 1024.0 / 1024, 2)  AS avg_size_mb
    FROM {meta(TABLE, 'files')}
    GROUP BY file_format
""")

,file_format,file_count,total_records,avg_records_per_file,min_records,max_records,total_size_bytes,avg_size_mb
0,PARQUET,43,11290285,262565.0,3600,6403928,83359422,1.85


---
## 6. Metadata log entries
---
The $metadata_log_entries table provides a view of metadata log entries of the Iceberg table.

You can retrieve the information about the metadata log entries of the Iceberg table test_table

by using the following query:

In [12]:
# The manifests for the current snapshot
manifests_df = query(f"""
    SELECT
        *
    FROM {meta(TABLE, 'metadata_log_entries')}
""")

print(f"Total manifests: {len(manifests_df)}")
manifests_df[:5]

Total manifests: 65


,timestamp,file,latest_snapshot_id,latest_schema_id,latest_sequence_number
0,2026-05-22 18:26:11.432000+00:00,s3://iceberg-warehouse/demo/transactions/metad...,NaN,NaN,NaN
1,2026-05-22 18:26:32.666000+00:00,s3://iceberg-warehouse/demo/transactions/metad...,5.200535e+18,0.0,1.0
2,2026-05-22 18:27:03.629000+00:00,s3://iceberg-warehouse/demo/transactions/metad...,5.170621e+18,0.0,2.0
3,2026-05-22 18:27:32.682000+00:00,s3://iceberg-warehouse/demo/transactions/metad...,2.524637e+18,0.0,3.0
4,2026-05-22 18:29:32.650000+00:00,s3://iceberg-warehouse/demo/transactions/metad...,1.381668e+18,0.0,4.0


---
## 7. Manifests (`$manifests`)
---

Manifest files are Avro files that group data files. The `$manifests` table exposes every manifest referenced by the current snapshot, including partition-level stats that let Iceberg prune manifests at planning time.

| Column | Type | Description |
|---|---|---|
| `path` | string | S3 path to the Avro manifest |
| `length` | bigint | Manifest file size in bytes |
| `partition_spec_id` | int | Partition spec used to write these files |
| `added_snapshot_id` | bigint | Snapshot that added this manifest |
| `added_data_files_count` | int | Data files added in this manifest |
| `existing_data_files_count` | int | Unchanged data files |
| `deleted_data_files_count` | int | Logically deleted files pending cleanup |
| `added_rows_count` | bigint | Rows added |
| `existing_rows_count` | bigint | Unchanged rows |
| `deleted_rows_count` | bigint | Deleted rows not yet compacted |

In [13]:
# The manifests for the current snapshot
manifests_df = query(f"""
    SELECT
        *
    FROM {meta(TABLE, 'manifests')}
    ORDER BY added_snapshot_id DESC
""")

print(f"Total manifests: {len(manifests_df)}")
manifests_df[:5]

Total manifests: 65


,path,length,partition_spec_id,added_snapshot_id,added_data_files_count,added_rows_count,existing_data_files_count,existing_rows_count,deleted_data_files_count,deleted_rows_count,partition_summaries
0,s3://iceberg-warehouse/demo/transactions/metad...,7501,0,9139992746180883843,1,14750,0,0,0,0,[]
1,s3://iceberg-warehouse/demo/transactions/metad...,7504,0,9119163870846862767,1,14800,0,0,0,0,[]
2,s3://iceberg-warehouse/demo/transactions/metad...,7493,0,9007986747489937402,1,14750,0,0,0,0,[]
3,s3://iceberg-warehouse/demo/transactions/metad...,7504,0,8946351738825895094,1,14800,0,0,0,0,[]
4,s3://iceberg-warehouse/demo/transactions/metad...,7501,0,8919139327694873599,1,14750,0,0,0,0,[]


In [14]:
# Manifest health — pending deletes indicate compaction is needed
query(f"""
    SELECT
        count(*)                        AS manifest_count,
        sum(added_data_files_count)     AS total_added_files,
        sum(existing_data_files_count)  AS total_existing_files,
        sum(deleted_data_files_count)   AS total_pending_deleted_files,
        sum(added_rows_count)           AS total_added_rows,
        sum(existing_rows_count)        AS total_existing_rows,
        sum(deleted_rows_count)         AS total_pending_deleted_rows
    FROM {meta(TABLE, 'manifests')}
""")

,manifest_count,total_added_files,total_existing_files,total_pending_deleted_files,total_added_rows,total_existing_rows,total_pending_deleted_rows
0,65,65,0,0,11615485,0,0


## 7.1 `all_manifests` vs `manifests`

The `$manifests` and `$all_manifests` tables provide a detailed overview of the manifests corresponding to
the snapshots performed in the log of the Iceberg table.

The `$manifests` table contains data for the current snapshot.
The `$all_manifests` table contains data for all snapshots.

In [15]:
# The all_manifests for the current snapshot
manifests_df = query(f"""
    SELECT
        *
    FROM {meta(TABLE, 'all_manifests')}
    ORDER BY added_snapshot_id DESC
""")

print(f"Total manifests: {len(manifests_df)}")
manifests_df[:5]

Total manifests: 2145


,path,length,partition_spec_id,added_snapshot_id,added_data_files_count,existing_data_files_count,deleted_data_files_count,partition_summaries
0,s3://iceberg-warehouse/demo/transactions/metad...,7501,0,9139992746180883843,1,0,0,[]
1,s3://iceberg-warehouse/demo/transactions/metad...,7501,0,9139992746180883843,1,0,0,[]
2,s3://iceberg-warehouse/demo/transactions/metad...,7501,0,9139992746180883843,1,0,0,[]
3,s3://iceberg-warehouse/demo/transactions/metad...,7501,0,9139992746180883843,1,0,0,[]
4,s3://iceberg-warehouse/demo/transactions/metad...,7501,0,9139992746180883843,1,0,0,[]


---
## 8. Partitions (`$partitions`)
---

The `$partitions` table provides a **partition-level summary** — record counts, file counts, and value bounds rolled up per partition value. Essential for spotting data skew.

| Column | Type | Description |
|---|---|---|
| `partition` | row | Partition column values |
| `record_count` | bigint | Total rows in this partition |
| `file_count` | bigint | Data files in this partition |
| `total_size` | bigint | Total compressed bytes |
| `data` | map | Per-column stats (null_count, nan_count, lower_bound, upper_bound) |

> If the table is unpartitioned, `$partitions` returns a single row.

In [16]:
# Partition-level stats
# For unpartitioned tables the 'partition' column is empty/absent — use SELECT * to get whatever columns exist
partitions_df = query(f"SELECT * FROM {meta(TABLE, 'partitions')}")

print(f"Columns : {list(partitions_df.columns)}")
print(f"Partitions: {len(partitions_df)}")
partitions_df

Columns : ['record_count', 'file_count', 'total_size', 'data']
Partitions: 1


,record_count,file_count,total_size,data
0,11615485,65,85397884,"(transaction_id: (min: 'txn-000001', max: 'txn..."


In [17]:
# Skew detection across partitions
if len(partitions_df) > 1:
    skew_df = partitions_df["record_count"].describe().to_frame()
    print("Record count distribution across partitions:")
    display(skew_df)
    skew_ratio = partitions_df["record_count"].max() / partitions_df["record_count"].min()
    print(f"\nMax/Min skew ratio: {skew_ratio:.1f}x")
    if skew_ratio > 10:
        print("WARNING: High data skew detected — consider repartitioning.")
    else:
        print("Partition distribution looks balanced.")
else:
    print("Table is unpartitioned or has a single partition.")

Table is unpartitioned or has a single partition.


---
## 9. Refs (`$refs`)
---

The `$refs` table shows all named references (branches and tags) for this table. With Apache Polaris, only the default `main` branch exists.

| Column | Type | Description |
|---|---|---|
| `name` | string | Reference name (`main`) |
| `type` | string | `BRANCH` or `TAG` |
| `snapshot_id` | bigint | Snapshot this reference points to |
| `max_reference_age_in_ms` | bigint | Retention policy |
| `min_snapshots_to_keep` | int | Minimum snapshots to retain |
| `max_snapshot_age_in_ms` | bigint | Snapshot age expiry threshold |

In [18]:
# Named references (branches / tags) via Iceberg $refs
refs_df = query(f"""
    SELECT
        *
    FROM {meta(TABLE, 'refs')}
""")

print(f"References (branches / tags): {len(refs_df)}")
refs_df

References (branches / tags): 1


,name,type,snapshot_id,max_reference_age_in_ms,min_snapshots_to_keep,max_snapshot_age_in_ms
0,main,BRANCH,5881717778084072778,None,None,None


---
## 10. Sample data & basic statistics
---

Quick sanity-check: read some rows and compute summary stats directly via Trino SQL.

In [19]:
# Sample 10 rows from the table
print(f"Sample rows from {FULL_TABLE}:")
query(f"SELECT * FROM {FULL_TABLE} LIMIT 10")

Sample rows from iceberg.demo.transactions:


,transaction_id,user_id,amount,currency,type,status,event_time
0,txn-816951,user-022,452.82,AUD,TRANSFER,FAILED,2026-05-22 18:57:02.039
1,txn-816952,user-045,876.42,AUD,TRANSFER,PENDING,2026-05-22 18:57:02.040
2,txn-816953,user-032,412.98,GBP,TRANSFER,COMPLETED,2026-05-22 18:57:02.040
3,txn-816958,user-023,150.93,CAD,TRANSFER,COMPLETED,2026-05-22 18:57:02.041
4,txn-816959,user-018,606.72,USD,DEPOSIT,REVERSED,2026-05-22 18:57:02.041
5,txn-816960,user-022,172.19,EUR,PURCHASE,FAILED,2026-05-22 18:57:02.041
6,txn-816954,user-013,833.61,GBP,REFUND,FAILED,2026-05-22 18:57:02.040
7,txn-816955,user-042,701.79,EUR,PURCHASE,COMPLETED,2026-05-22 18:57:02.041
8,txn-816956,user-046,985.19,CAD,WITHDRAWAL,PENDING,2026-05-22 18:57:02.041
9,txn-816957,user-044,1085.20,AUD,WITHDRAWAL,COMPLETED,2026-05-22 18:57:02.041


In [20]:
# Table-specific statistics — transactions
if TABLE == "transactions":
    print("Transactions — summary statistics:")
    display(query(f"""
        SELECT
            count(*)                        AS total_rows,
            count(DISTINCT user_id)         AS distinct_users,
            round(avg(amount), 2)           AS avg_amount,
            min(amount)                     AS min_amount,
            max(amount)                     AS max_amount,
            min(event_time)                 AS earliest_event,
            max(event_time)                 AS latest_event
        FROM {FULL_TABLE}
    """))

    print("\nBreakdown by status:")
    display(query(f"""
        SELECT
            status,
            count(*)                    AS txn_count,
            round(sum(amount), 2)       AS total_amount
        FROM {FULL_TABLE}
        GROUP BY status
        ORDER BY txn_count DESC
    """))

    print("\nBreakdown by type:")
    display(query(f"""
        SELECT
            type,
            count(*)                    AS txn_count,
            round(avg(amount), 2)       AS avg_amount
        FROM {FULL_TABLE}
        GROUP BY type
        ORDER BY txn_count DESC
    """))

elif TABLE == "users":
    print("Users — summary statistics:")
    display(query(f"""
        SELECT
            count(*)                    AS total_users,
            count(DISTINCT country)     AS distinct_countries,
            min(created_at)             AS first_user,
            max(created_at)             AS latest_user
        FROM {FULL_TABLE}
    """))

    print("\nTop 10 countries by user count:")
    display(query(f"""
        SELECT
            country,
            count(*) AS user_count
        FROM {FULL_TABLE}
        GROUP BY country
        ORDER BY user_count DESC
        LIMIT 10
    """))

Transactions — summary statistics:


,total_rows,distinct_users,avg_amount,min_amount,max_amount,earliest_event,latest_event
0,11615485,50,1000.5,1.0,2000.0,2026-05-14 18:26:55.065,2026-05-22 19:00:02.259



Breakdown by status:


,status,txn_count,total_amount
0,REVERSED,2905081,2.905275e+09
1,PENDING,2904467,2.907728e+09
2,FAILED,2903333,2.905330e+09
3,COMPLETED,2902604,2.902990e+09



Breakdown by type:


,type,txn_count,avg_amount
0,REFUND,2329079,1000.27
1,PURCHASE,2326441,1000.81
2,WITHDRAWAL,2326138,1000.90
3,DEPOSIT,2326030,1000.26
4,TRANSFER,2322447,1000.28


---
## 11. Table health dashboard
---

Combine all metadata sources into a single health summary — snapshot count, file sizes, manifest bloat, and pending deletes.

In [21]:
# ── Snapshot count ──────────────────────────────────────────────────────────
snap_row = query(f"SELECT count(*) AS cnt FROM {meta(TABLE, 'history')}").iloc[0]
snapshot_count = snap_row["cnt"]

# ── File stats ──────────────────────────────────────────────────────────────
file_row = query(f"""
    SELECT
        count(*)                                            AS file_count,
        sum(record_count)                                   AS total_records,
        round(avg(record_count), 0)                         AS avg_records_per_file,
        round(avg(file_size_in_bytes) / 1024.0 / 1024, 2)  AS avg_size_mb,
        sum(file_size_in_bytes)                             AS total_size_bytes
    FROM {meta(TABLE, 'files')}
    WHERE content = 0
""").iloc[0]

# ── Manifest bloat ──────────────────────────────────────────────────────────
manifest_row = query(f"""
    SELECT
        count(*)                        AS manifest_count,
        sum(deleted_data_files_count)   AS pending_deleted_files,
        sum(deleted_rows_count)         AS pending_deleted_rows
    FROM {meta(TABLE, 'manifests')}
""").iloc[0]

# ── Print summary ───────────────────────────────────────────────────────────
SEP = "=" * 56
print(SEP)
print(f"  TABLE HEALTH: {FULL_TABLE}")
print(SEP)
print(f"  Snapshots committed        : {snapshot_count}")
print(f"  Current data files         : {file_row['file_count']}")
print(f"  Total records              : {int(file_row['total_records'] or 0):,}")
print(f"  Avg records per file       : {int(file_row['avg_records_per_file'] or 0):,}")
print(f"  Avg file size              : {file_row['avg_size_mb']} MB")
print(f"  Total data size            : {round((file_row['total_size_bytes'] or 0) / 1024 / 1024, 2)} MB")
print(f"  Manifests                  : {manifest_row['manifest_count']}")
print(f"  Pending deleted files      : {manifest_row['pending_deleted_files'] or 0}")
print(f"  Pending deleted rows       : {int(manifest_row['pending_deleted_rows'] or 0):,}")
print()
print("  Recommendations:")
avg_recs = int(file_row["avg_records_per_file"] or 0)
if avg_recs < 10_000:
    print("  [!] Low records-per-file — consider running RewriteDataFiles compaction")
else:
    print("  [ok] File sizes look healthy")
pending_del = int(manifest_row["pending_deleted_files"] or 0)
if pending_del > 0:
    print("  [!] Pending deletes — run expireSnapshots to reclaim storage")
else:
    print("  [ok] No pending deletes")
print(SEP)

  TABLE HEALTH: iceberg.demo.transactions
  Snapshots committed        : 66
  Current data files         : 66.0
  Total records              : 11,630,135
  Avg records per file       : 176,214
  Avg file size              : 1.24 MB
  Total data size            : 81.53 MB
  Manifests                  : 66
  Pending deleted files      : 0
  Pending deleted rows       : 0

  Recommendations:
  [ok] File sizes look healthy
  [ok] No pending deletes


---
## 12 Trino cluster info
---

Inspect the Trino cluster itself: active workers, running queries, catalog configuration.

In [22]:
import urllib.request, json

# Trino REST API — cluster info
with urllib.request.urlopen("http://trino-coordinator:8080/v1/info") as r:
    info = json.loads(r.read())
print("Trino cluster info:")
for k, v in info.items():
    print(f"  {k}: {v}")

Trino cluster info:
  nodeVersion: {'version': '468'}
  environment: production
  coordinator: True
  starting: False
  uptime: 35.33m


In [23]:
# Active worker nodes
# /v1/node requires the X-Trino-User header (unlike /v1/info which is public)
req = urllib.request.Request(
    "http://trino-coordinator:8080/v1/node",
    headers={"X-Trino-User": "jupyter"},
)
with urllib.request.urlopen(req) as r:
    nodes = json.loads(r.read())

nodes_df = pd.DataFrame([
    {
        "node_id": n.get("nodeId"),
        "uri": n.get("uri"),
        "state": n.get("state"),
        "coordinator": n.get("coordinator"),
    }
    for n in nodes
])
print(f"Active nodes: {len(nodes_df)}")
nodes_df

Active nodes: 1


,node_id,uri,state,coordinator
0,None,http://172.20.0.10:8080/v1/status,None,None


In [40]:
# Available catalogs
print("Catalogs available in Trino:")
query("SHOW CATALOGS")

Catalogs available in Trino:


,Catalog
0,iceberg
1,jmx
2,memory
3,system
4,tpcds
5,tpch


In [24]:
# Schemas in the iceberg catalog
print("Schemas in the iceberg catalog:")
query("SHOW SCHEMAS IN iceberg")

Schemas in the iceberg catalog:


,Schema
0,demo
1,information_schema
